In [1]:
import sys

print(sys.version)

3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/data.csv")

In [3]:
print(df.shape)

(106, 10)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Class   106 non-null    str    
 1   I0      106 non-null    float64
 2   PA500   106 non-null    float64
 3   HFS     106 non-null    float64
 4   DA      106 non-null    float64
 5   Area    106 non-null    float64
 6   A.DA    106 non-null    float64
 7   Max.IP  106 non-null    float64
 8   DR      106 non-null    float64
 9   P       106 non-null    float64
dtypes: float64(9), str(1)
memory usage: 8.4 KB


In [5]:
print(df.dtypes)

Class         str
I0        float64
PA500     float64
HFS       float64
DA        float64
Area      float64
A.DA      float64
Max.IP    float64
DR        float64
P         float64
dtype: object


In [6]:
print(df.isnull().sum())

Class     0
I0        0
PA500     0
HFS       0
DA        0
Area      0
A.DA      0
Max.IP    0
DR        0
P         0
dtype: int64


In [7]:
X = df.drop(columns=["Class"])
y = df["Class"]

In [8]:
print("X:", X.shape)
print("y:", y.shape)

X: (106, 9)
y: (106,)


In [9]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

In [10]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [11]:
print("Числовые признаки:")
print(numeric_features)

print("\nКатегориальные признаки:")
print(categorical_features)

Числовые признаки:
['I0', 'PA500', 'HFS', 'DA', 'Area', 'A.DA', 'Max.IP', 'DR', 'P']

Категориальные признаки:
[]


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (84, 9)
X_test: (22, 9)
y_train: (84,)
y_test: (22,)


In [14]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [15]:
SimpleImputer(strategy="median")

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [16]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [18]:
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['I0', 'PA500', 'HFS', 'DA', 'Area', 'A.DA', 'Max.IP', 'DR', 'P']
Categorical: []


In [19]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [20]:
X_test_transformed = preprocessor.transform(X_test)

In [21]:
print("Исходный X_train:", X_train.shape)
print("После preprocessing:", X_train_transformed.shape)

print("Исходный X_test:", X_test.shape)
print("После preprocessing:", X_test_transformed.shape)

Исходный X_train: (84, 9)
После preprocessing: (84, 9)
Исходный X_test: (22, 9)
После preprocessing: (22, 9)


In [22]:
print("Пропуски в X_train:",
      np.isnan(X_train_transformed).sum())

print("Пропуски в X_test:",
      np.isnan(X_test_transformed).sum())

Пропуски в X_train: 0
Пропуски в X_test: 0


In [23]:
print(
    "Количество признаков после preprocessing:",
    X_train_transformed.shape[1]
)

Количество признаков после preprocessing: 9


In [24]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__I0' 'num__PA500' 'num__HFS' 'num__DA' 'num__Area' 'num__A.DA'
 'num__Max.IP' 'num__DR' 'num__P']


### Проверка отсутствия утечки данных

Для предотвращения утечки данных preprocessing обучается только на обучающей выборке X_train с помощью метода fit_transform(). Тестовая выборка X_test не используется при вычислении медиан, параметров масштабирования или категорий.

После обучения preprocessing к тестовой выборке применяется только метод transform(). Таким образом, информация из тестовой выборки не влияет на параметры предварительной обработки.

In [25]:
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]

print("Медианы, рассчитанные на train:")
print(imputer.statistics_)

Медианы, рассчитанные на train:
[3.89936489e+02 1.05854219e-01 8.86627260e-02 1.19639412e+02
 1.85992004e+03 1.61336567e+01 4.42160397e+01 9.32619682e+01
 4.58550747e+02]


In [26]:
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]

print("Средние значения train:")
print(scaler.mean_)

print("\nСтандартные отклонения train:")
print(scaler.scale_)

Средние значения train:
[7.81167626e+02 1.20951931e-01 1.15901758e-01 1.91487053e+02
 7.59598203e+03 2.33593364e+01 7.38198284e+01 1.68451024e+02
 8.02838759e+02]

Стандартные отклонения train:
[7.52848439e+02 7.01892469e-02 1.00007063e-01 1.99303354e+02
 2.03727641e+04 2.43478766e+01 7.96510538e+01 1.90771328e+02
 7.54207919e+02]


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# 1. Загрузка данных
df = pd.read_csv("../data/data.csv")

# 2. Разделение признаков и целевой переменной
X = df.drop(columns=["Class"])
y = df["Class"]

# 3. Определение типов признаков
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Числовые признаки:")
print(numeric_features)

print("\nКатегориальные признаки:")
print(categorical_features)

# 4. Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. Обработка числовых признаков
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# 6. Обработка категориальных признаков
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

# 7. ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 8. Обучение только на train
X_train_transformed = preprocessor.fit_transform(X_train)

# 9. Применение к test без повторного обучения
X_test_transformed = preprocessor.transform(X_test)

# 10. Проверка размерностей
print("\nРазмерности:")
print("X_train:", X_train.shape)
print("X_train_transformed:", X_train_transformed.shape)

print("X_test:", X_test.shape)
print("X_test_transformed:", X_test_transformed.shape)

# 11. Названия итоговых признаков
feature_names = preprocessor.get_feature_names_out()

print("\nИтоговые признаки:")
print(feature_names)

Числовые признаки:
['I0', 'PA500', 'HFS', 'DA', 'Area', 'A.DA', 'Max.IP', 'DR', 'P']

Категориальные признаки:
[]

Размерности:
X_train: (84, 9)
X_train_transformed: (84, 9)
X_test: (22, 9)
X_test_transformed: (22, 9)

Итоговые признаки:
['num__I0' 'num__PA500' 'num__HFS' 'num__DA' 'num__Area' 'num__A.DA'
 'num__Max.IP' 'num__DR' 'num__P']


## 1. Классификация признаков

После анализа структуры датасета признаки были разделены по типам.

Числовые признаки:

- I0
- PA500
- HFS
- DA
- Area
- A.DA
- Max.IP
- DR
- P

Категориальных признаков среди входных признаков нет.

Целевая переменная Class была отделена от входных признаков и не включалась в preprocessing.

In [8]:
print("Количество пропусков по каждому признаку:")
print(X.isnull().sum())

print("\nВсего пропусков:")
print(X.isnull().sum().sum())

Количество пропусков по каждому признаку:
I0        0
PA500     0
HFS       0
DA        0
Area      0
A.DA      0
Max.IP    0
DR        0
P         0
dtype: int64

Всего пропусков:
0


## Анализ пропущенных значений

В датасете отсутствуют пропущенные значения. Для всех 9 числовых признаков количество пропусков равно 0:

- I0 — 0
- PA500 — 0
- HFS — 0
- DA — 0
- Area — 0
- A.DA — 0
- Max.IP — 0
- DR — 0
- P — 0

Общее количество пропусков: 0.

Несмотря на отсутствие пропусков в текущем наборе данных, в pipeline для числовых признаков предусмотрено заполнение пропусков медианой с помощью SimpleImputer(strategy="median"). Это делает pipeline устойчивым к возможному появлению пропусков в новых данных.

## Обоснование стратегии заполнения

Все входные признаки являются числовыми. Для них выбрана стратегия заполнения пропусков медианой:

SimpleImputer(strategy="median").

Медиана выбрана потому, что она менее чувствительна к выбросам, чем среднее арифметическое. Для числовых измерений это позволяет получить более устойчивое значение при наличии отдельных аномально больших или маленьких наблюдений.

В текущем датасете пропуски отсутствуют, поэтому фактическая замена значений не выполняется. Однако стратегия median остаётся частью pipeline для корректной обработки возможных пропусков в будущем.

Категориальные признаки отсутствуют, поэтому отдельная категория "missing" и OneHotEncoder фактически не применяются.

In [9]:
print("Размер X_train до обработки:", X_train.shape)
print("Размер X_train после обработки:", X_train_transformed.shape)

print("\nРазмер X_test до обработки:", X_test.shape)
print("Размер X_test после обработки:", X_test_transformed.shape)

print("\nКоличество признаков после обработки:",
      len(preprocessor.get_feature_names_out()))

Размер X_train до обработки: (84, 9)
Размер X_train после обработки: (84, 9)

Размер X_test до обработки: (22, 9)
Размер X_test после обработки: (22, 9)

Количество признаков после обработки: 9


In [10]:
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]

print("Медианы, рассчитанные на X_train:")
print(imputer.statistics_)

Медианы, рассчитанные на X_train:
[3.89936489e+02 1.05854219e-01 8.86627260e-02 1.19639412e+02
 1.85992004e+03 1.61336567e+01 4.42160397e+01 9.32619682e+01
 4.58550747e+02]


In [11]:
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]

print("Средние значения, рассчитанные на X_train:")
print(scaler.mean_)

print("\nСтандартные отклонения, рассчитанные на X_train:")
print(scaler.scale_)

Средние значения, рассчитанные на X_train:
[7.81167626e+02 1.20951931e-01 1.15901758e-01 1.91487053e+02
 7.59598203e+03 2.33593364e+01 7.38198284e+01 1.68451024e+02
 8.02838759e+02]

Стандартные отклонения, рассчитанные на X_train:
[7.52848439e+02 7.01892469e-02 1.00007063e-01 1.99303354e+02
 2.03727641e+04 2.43478766e+01 7.96510538e+01 1.90771328e+02
 7.54207919e+02]


In [12]:
preprocessor.fit_transform(X_train)

array([[-6.46468472e-01,  1.22091427e+00,  3.51821633e+00,
         1.69790516e-02, -1.00856511e-01,  2.08485027e-01,
        -4.65204538e-01,  1.22418115e-01, -4.73775799e-01],
       [-1.18477491e-01, -1.35272180e+00, -2.93313559e-01,
        -4.06597027e-03, -3.57914678e-01, -8.93860076e-01,
        -6.25780934e-01,  1.08571383e-01, -2.76479639e-01],
       [-5.69186514e-01,  1.49108901e-02, -2.51428616e-01,
        -6.16944996e-01, -3.20517341e-01, -3.20410399e-01,
        -3.78248647e-01, -6.06266196e-01, -5.57015594e-01],
       [-8.08465286e-01,  8.95090371e-02, -7.74990413e-01,
        -7.72407373e-01, -3.63414794e-01, -7.49120019e-01,
        -6.84206237e-01, -7.14264584e-01, -8.32535662e-01],
       [-7.71958332e-01, -1.18611928e+00,  1.38426954e-02,
        -7.48458945e-01, -3.62011315e-01, -7.45086634e-01,
        -7.92758680e-01, -6.68355895e-01, -7.75388896e-01],
       [ 1.48613229e+00, -9.47405206e-01, -2.10614245e-02,
         4.07072805e-01, -5.61477850e-03,  1.677421

## Проверка отсутствия утечки данных

Для предотвращения утечки данных preprocessing был разделён на два этапа.

Для обучающей выборки использовалась операция:

fit_transform(X_train)

На этом этапе pipeline обучается и рассчитывает необходимые параметры только на обучающих данных.

Для тестовой выборки использовалась операция:

transform(X_test)

Тестовая выборка не использовалась при обучении preprocessing. Поэтому информация из X_test не влияет на вычисление параметров заполнения пропусков и стандартизации.

Размерности до и после обработки:

| Выборка | До обработки | После обработки |
|---|---:|---:|
| Train | 84 × 9 | 84 × 9 |
| Test | 22 × 9 | 22 × 9 |

Количество строк сохранилось, а количество признаков осталось равным 9. Это связано с тем, что все входные признаки являются числовыми и категориальные признаки отсутствуют.